In [ ]:
import pandas as pd

portfolio = pd.read_csv("portfolio.csv")
profile = pd.read_csv("profile.csv")
transcript = pd.read_csv("transcript.csv")

print(portfolio.head())
print(profile.head())
print(transcript.head())

In [ ]:
# Flag and check how many "fake" profiles exist
fake_profiles = profile[profile['age'] == 118]
print(f"Fake/missing profiles: {len(fake_profiles)} out of {len(profile)}")

# Drop them for now (we can revisit later if needed)
profile_clean = profile[profile['age'] != 118].copy()

In [ ]:
import ast

def safe_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {}
    return {}

transcript['value_dict'] = transcript['value'].apply(safe_eval)

transcript['offer_id'] = transcript['value_dict'].apply(lambda x: x.get('offer id') or x.get('offer_id'))
transcript['amount'] = transcript['value_dict'].apply(lambda x: x.get('amount'))
transcript['reward_received'] = transcript['value_dict'].apply(lambda x: x.get('reward'))

print(transcript['event'].value_counts())
print(transcript.head())

In [ ]:
# Total spend per person (from transaction events only)
spend_per_person = transcript[transcript['event'] == 'transaction'].groupby('person')['amount'].sum().reset_index()
spend_per_person.columns = ['person', 'total_spend']

# Did each person complete at least one offer?
completers = transcript[transcript['event'] == 'offer completed']['person'].unique()
spend_per_person['completed_offer'] = spend_per_person['person'].isin(completers)

print(spend_per_person['completed_offer'].value_counts())
print(spend_per_person.groupby('completed_offer')['total_spend'].mean())

In [ ]:
# For each person, check the order of events per offer they received
viewed = set(transcript[transcript['event'] == 'offer viewed']['person'] + transcript[transcript['event'] == 'offer viewed']['offer_id'].astype(str))
completed = transcript[transcript['event'] == 'offer completed'].copy()
completed['person_offer'] = completed['person'] + completed['offer_id'].astype(str)

completed['was_viewed'] = completed['person_offer'].isin(viewed)
print(completed['was_viewed'].value_counts())

In [ ]:
# Get the set of "person_offer" combos for each category
blind_completers = set(completed[completed['was_viewed'] == False]['person_offer'])
real_completers = set(completed[completed['was_viewed'] == True]['person'])
blind_people = set(completed[completed['was_viewed'] == False]['person'])

spend_per_person['real_completer'] = spend_per_person['person'].isin(real_completers)
spend_per_person['blind_completer'] = spend_per_person['person'].isin(blind_people) & ~spend_per_person['person'].isin(real_completers)

def group_label(row):
    if row['real_completer']:
        return 'viewed_then_completed'
    elif row['blind_completer']:
        return 'blind_completion_only'
    else:
        return 'no_real_completion'

spend_per_person['group'] = spend_per_person.apply(group_label, axis=1)

print(spend_per_person['group'].value_counts())
print(spend_per_person.groupby('group')['total_spend'].mean())

In [ ]:
# Get the time each person first viewed an offer
first_view = transcript[transcript['event'] == 'offer viewed'].groupby('person')['time'].min().reset_index()
first_view.columns = ['person', 'first_view_time']

# Get all transactions with their timestamp
transactions = transcript[transcript['event'] == 'transaction'][['person', 'time', 'amount']]

# Merge to know each transaction's timing relative to first offer view
merged = transactions.merge(first_view, on='person', how='inner')

merged['period'] = merged.apply(
    lambda row: 'before' if row['time'] < row['first_view_time'] else 'after', axis=1
)

before_after = merged.groupby(['person', 'period'])['amount'].sum().unstack(fill_value=0)
print(before_after.mean())

In [ ]:
# Get how many days each period actually covers, per person
first_view_days = first_view.copy()
first_view_days['first_view_time'] = first_view_days['first_view_time']

# Total observation window is time 0 to max time in dataset
max_time = transcript['time'].max()

merged_days = first_view.copy()
merged_days['before_days'] = merged_days['first_view_time'] / 24  # time is in hours
merged_days['after_days'] = (max_time - merged_days['first_view_time']) / 24

# Merge spend numbers back in
spend_summary = before_after.reset_index().merge(merged_days, on='person')

# Spend PER DAY, not total spend
spend_summary['before_per_day'] = spend_summary['before'] / spend_summary['before_days'].replace(0, 0.5)
spend_summary['after_per_day'] = spend_summary['after'] / spend_summary['after_days'].replace(0, 0.5)

print(spend_summary[['before_per_day', 'after_per_day']].mean())

In [ ]:
# Total reward paid out (from completed offers)
total_reward_paid = transcript[transcript['event'] == 'offer completed']['reward_received'].sum()

# Total extra spend estimate: (after_per_day - before_per_day) * average days in "after" period * number of people
avg_after_days = spend_summary['after_days'].mean()
n_people = len(spend_summary)
extra_spend_per_day = spend_summary['after_per_day'].mean() - spend_summary['before_per_day'].mean()

estimated_extra_revenue = extra_spend_per_day * avg_after_days * n_people

print(f"Total reward paid out: ${total_reward_paid:,.2f}")
print(f"Estimated extra revenue generated: ${estimated_extra_revenue:,.2f}")
print(f"Return ratio (revenue per $1 of reward): {estimated_extra_revenue / total_reward_paid:.2f}")

Part 2

In [ ]:
# Merge everything into one clean table
profile_clean['became_member_on'] = pd.to_datetime(profile_clean['became_member_on'], format='%Y%m%d')
profile_clean['membership_days'] = (pd.Timestamp('2018-12-31') - profile_clean['became_member_on']).dt.days

# Treatment: did they view before completing? (from your earlier group split)
spend_per_person['treatment'] = (spend_per_person['group'] == 'viewed_then_completed').astype(int)

# Merge with spend_summary (has after_per_day) and profile_clean (has confounders)
causal_df = spend_per_person.merge(spend_summary[['person', 'after_per_day']], on='person', how='inner')
causal_df = causal_df.merge(profile_clean[['id', 'age', 'income', 'membership_days']], left_on='person', right_on='id', how='inner')

causal_df = causal_df[['person', 'treatment', 'after_per_day', 'age', 'income', 'membership_days']].dropna()
print(causal_df.shape)
print(causal_df.head())

In [ ]:
!pip install dowhy

In [ ]:
from dowhy import CausalModel

model = CausalModel(
    data=causal_df,
    treatment='treatment',
    outcome='after_per_day',
    common_causes=['age', 'income', 'membership_days']
)

model.view_model()

In [ ]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

In [ ]:
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression"
)
print(estimate)

In [ ]:
# Test 1: add a random/fake confounder - if the estimate barely changes, that's reassuring
refute1 = model.refute_estimate(identified_estimand, estimate, method_name="random_common_cause")
print(refute1)

# Test 2: replace treatment with a placebo (fake/random treatment) - estimate should collapse toward 0
refute2 = model.refute_estimate(identified_estimand, estimate, method_name="placebo_treatment_refuter")
print(refute2)